# CLIMADA

In [ ]:
import sim_mel as sm
from climada.hazard import Hazard, Centroids, TCTracks, TropCyclone
from climada.entity import Exposures
from climada.entity.exposures import LitPop
from climada.entity.impact_funcs import ImpactFuncSet, ImpfTropCyclone
# from climada.entity import Measure, MeasureSet, Entity
from climada.engine import ImpactCalc, Impact
# import os
# from climada.util import save, load
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import collections
import climada.util.coordinates as u_coord
# import climada.entity.exposures.litpop as lp
from datetime import datetime as dt
import geopandas as gpd
from scipy import sparse as sp

In [ ]:
## set parameters
# Exposure
my_country = 'AUS'
m, n = 1,1 # based on analysis by Eberenz et al. (2020), (1,1) is the best combination.
my_fin_mode = 'pc'
ref_year = 2023

# Hazard
start_year = 1980
end_year = 2023
n_synth_tracks = 300

my_res_arcsec = 600
n_sim = 1000000 # 1million
starting_phase = 'Nino'

## Exposure

In [ ]:
# Exposure
exp = LitPop.from_countries(my_country, res_arcsec=my_res_arcsec, reference_year=ref_year, exponents =(m,n), fin_mode=my_fin_mode)
exp.check()

In [ ]:
# check
print(f'Total exposure value to {my_country}: USD {exp.gdf.value.sum():,.2f} / AUD {exp.gdf.value.sum()/0.6630:,.2f} ({ref_year})')

## Hazard

In [ ]:
# Hazard
haz = TropCyclone.from_hdf5('Hazards/haz_aus_300synth.hdf5')
haz.check()
exp.assign_centroids(haz)

## Impact

In [ ]:
impf_tc = ImpfTropCyclone.from_emanuel_usa()

impf_set = ImpactFuncSet([impf_tc])
impf_set.check()

## Results
### 1-year losses

In [ ]:
loss_orig = sm.sim_losses_per_point(haz, exp, impf_set, 'orig', n_sim, n_years=1, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_orig.npz', loss_orig)

In [ ]:
loss_MMNHPP_FFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = False, intensity = False, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_FFF.npz', loss_MMNHPP_FFF)

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_TTT_50.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, p_loc = 0.8, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_TTT_80.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, p_loc = 1, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_TTT_100.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_FTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = False, intensity = True, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_FTF.npz', loss_MMNHPP_FTF)

In [ ]:
loss_MMNHPP_FFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = False, intensity = False, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_FFT.npz', loss_MMNHPP_FFT)

In [ ]:
loss_MMNHPP_TFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, intensity = False, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_TFF_50.npz', loss_MMNHPP_TFF)

In [ ]:
loss_MMNHPP_TTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, loc = True, intensity = True, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_TTF_50.npz', loss_MMNHPP_TTF)

In [ ]:
loss_MMNHPP_TFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, loc = True, intensity = False, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_TFT_50.npz', loss_MMNHPP_TFT)

In [ ]:
loss_MMNHPP_FTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, loc = False, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 1yr/loss_MMNHPP_FTT.npz', loss_MMNHPP_FTT)

### 5-year losses

In [ ]:
loss_orig = sm.sim_losses_per_point(haz, exp, impf_set, 'orig', n_sim, n_years=5, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_orig.npz', loss_orig)

In [ ]:
loss_MMNHPP_FFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = False, intensity = False, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_FFF.npz', loss_MMNHPP_FFF)

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = True, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_TTT_50.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = True, p_loc = 0.8, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_TTT_80.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = True, p_loc = 1, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_TTT_100.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_TFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = True, intensity = False, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_TFF_50.npz', loss_MMNHPP_TFF)

In [ ]:
loss_MMNHPP_FTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = False, intensity = True, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_FTF.npz', loss_MMNHPP_FTF)

In [ ]:
loss_MMNHPP_FFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = False, intensity = False, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_FFT.npz', loss_MMNHPP_FFT)

In [ ]:
loss_MMNHPP_TTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = True, intensity = True, damage = False, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_TTF_50.npz', loss_MMNHPP_TTF)

In [ ]:
loss_MMNHPP_TFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = True, intensity = False, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_TFT_50.npz', loss_MMNHPP_TFT)

In [ ]:
loss_MMNHPP_FTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=5, loc = False, intensity = True, damage = True, num_cpus=8)
sp.save_npz('Outputs/Loss outputs 5yr/loss_MMNHPP_FTT.npz', loss_MMNHPP_FTT)